In [10]:
# Lucky Choudhary

import pandas as pd
import os

# Load all dataset 
train = pd.read_csv("../data/raw/train.csv")
features = pd.read_csv("../data/raw/features.csv")
stores = pd.read_csv("../data/raw/stores.csv")

# shape Helps to identify the rows and columns in datasets in order(row,column)
print("Train:", train.shape)
print("Features:", features.shape)
print("Stores:", stores.shape)

Train: (421570, 5)
Features: (8190, 12)
Stores: (45, 3)


In [11]:
# merge Helps to merge all load file from before by using selective columns such as Store and Date to avoide any missing data 
df = pd.merge(train, features, on=['Store', 'Date'], how='left')
df = pd.merge(df, stores, on='Store', how='left')

# head Prints first five rows of given dataset 
print("Merged shape:", df.shape)
print(df.head())

Merged shape: (421570, 17)
   Store  Dept        Date  Weekly_Sales  IsHoliday_x  Temperature  \
0      1     1  2010-02-05      24924.50        False        42.31   
1      1     1  2010-02-12      46039.49         True        38.51   
2      1     1  2010-02-19      41595.55        False        39.93   
3      1     1  2010-02-26      19403.54        False        46.63   
4      1     1  2010-03-05      21827.90        False        46.50   

   Fuel_Price  MarkDown1  MarkDown2  MarkDown3  MarkDown4  MarkDown5  \
0       2.572        NaN        NaN        NaN        NaN        NaN   
1       2.548        NaN        NaN        NaN        NaN        NaN   
2       2.514        NaN        NaN        NaN        NaN        NaN   
3       2.561        NaN        NaN        NaN        NaN        NaN   
4       2.625        NaN        NaN        NaN        NaN        NaN   

          CPI  Unemployment  IsHoliday_y Type    Size  
0  211.096358         8.106        False    A  151315  
1  211.

In [12]:
# After completation of merging of datasets check columns first to avoide any duplicating columns
print(df.columns)

# Fix duplicate holiday columns if present because this part of code only work once and then useless
# First check for possible duplicate columns in this case 'IsHoliday_x' and 'IsHoliday_y'
if 'IsHoliday_x' in df.columns and 'IsHoliday_y' in df.columns:
# If find then replace 'IsHoliday_x' to create 'IsHoliday' column   
    df['IsHoliday'] = df['IsHoliday_x']
    # drop helps to delete 'IsHoliday_x' and 'IsHoliday_y'
    df.drop(['IsHoliday_x', 'IsHoliday_y'], axis=1, inplace=True)
    print("Duplicate holiday columns cleaned")
else:
    print("Only one IsHoliday column exists")

Index(['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday_x', 'Temperature',
       'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4',
       'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday_y', 'Type', 'Size'],
      dtype='str')
Duplicate holiday columns cleaned


In [17]:
# Convert text date into date formate so it can be used in future analysis
df['Date'] = pd.to_datetime(df['Date'])

# Create new columns year,month,month_name and week 
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.month_name()
df['Week'] = df['Date'].dt.isocalendar().week

print(df[['Date', 'Year', 'Month', 'Month_Name']].head())

        Date  Year  Month Month_Name
0 2010-02-05  2010      2   February
1 2010-02-12  2010      2   February
2 2010-02-19  2010      2   February
3 2010-02-26  2010      2   February
4 2010-03-05  2010      3      March


In [18]:
# Checks for duplicate rows 
print("Duplicates:", df.duplicated().sum())

# Checks for missing columns value
print("\nMissing Values:\n")
print(df.isnull().sum())

Duplicates: 0

Missing Values:

Store                0
Dept                 0
Date                 0
Weekly_Sales         0
Temperature          0
Fuel_Price           0
MarkDown1       270889
MarkDown2       310322
MarkDown3       284479
MarkDown4       286603
MarkDown5       270138
CPI                  0
Unemployment         0
Type                 0
Size                 0
IsHoliday            0
Year                 0
Month                0
Month_Name           0
Week                 0
dtype: int64


In [15]:
# Top stores
print(df.groupby('Store')['Weekly_Sales'].sum().sort_values(ascending=False).head())

# Top departments
print(df.groupby('Dept')['Weekly_Sales'].sum().sort_values(ascending=False).head())

# Holiday impact
print(df.groupby('IsHoliday')['Weekly_Sales'].mean())

# Monthly trend
print(df.groupby('Month')['Weekly_Sales'].mean())

Store
20    3.013978e+08
4     2.995440e+08
14    2.889999e+08
13    2.865177e+08
2     2.753824e+08
Name: Weekly_Sales, dtype: float64
Dept
92    4.839433e+08
95    4.493202e+08
38    3.931181e+08
72    3.057252e+08
90    2.910685e+08
Name: Weekly_Sales, dtype: float64
IsHoliday
False    15901.445069
True     17035.823187
Name: Weekly_Sales, dtype: float64
Month
1     14126.075111
2     16008.779217
3     15416.657597
4     15650.338357
5     15776.337202
6     16326.137002
7     15861.419650
8     16062.516933
9     15095.886154
10    15243.855576
11    17491.031424
12    19355.702141
Name: Weekly_Sales, dtype: float64


In [16]:
# Ensure folder exists in given location
os.makedirs("../data/processed", exist_ok=True)

# Save file if don't exist at given location
df.to_csv("../data/processed/cleaned_walmart.csv", index=False)

print("File saved successfully")

File saved successfully


## Key Insights

- Certain stores generate significantly higher sales than others
- Holiday periods show noticeable increase in sales
- Some departments consistently outperform others
- Monthly trend indicates seasonal sales patterns